In [ ]:
import os, sys, time, random
import numpy as np
import pandas as pd
import lightgbm as lgbm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

random.seed(42)
np.random.seed(42)
os.environ["PYTHONHASHSEED"] = "42"

print(os.listdir("../input"))




In [ ]:
_month_map = {
    1: "jan",
    2: "feb",
    3: "mar",
    4: "apr",
    5: "may",
    6: "jun",
    7: "jul",
    8: "aug",
    9: "sep",
    10: "oct",
    11: "nov",
    12: "dec",
}
_quarter_map = {0: "Q1", 1: "Q2", 2: "Q3", 3: "Q4"}  # based on hour // 6
_week_bins = [0, 7, 14, 21, 31]
_week_labels = ["1W", "2W", "3W", "4W"]


def _week_translation(day_series):
    """Vectorized week bucket → label."""
    return pd.cut(
        day_series,
        bins=_week_bins,
        labels=_week_labels,
        right=True,
        include_lowest=True,
    ).astype(str)


def timeExpansion(timeSeries):
    """
    Build dummy columns for year, month, week bucket and quarter.
    Uses sparse one‑hot encoding to minimise memory overhead.
    """
    years = timeSeries.dt.year
    months = timeSeries.dt.month.map(_month_map)
    days = _week_translation(timeSeries.dt.day)
    quarters = (timeSeries.dt.hour // 6).map(_quarter_map)

    dummy0 = pd.DataFrame(
        {"year": years, "month": months, "day": days, "quarter": quarters},
        index=timeSeries.index,
    )
    return pd.get_dummies(dummy0, dtype=np.float32, sparse=True)


def add_travel_vector_features(df):
    """Exact replica of the original feature creation."""
    df["abs_diff_longitude"] = (df.dropoff_longitude - df.pickup_longitude).abs()
    df["abs_diff_latitude"] = (df.dropoff_latitude - df.pickup_latitude).abs()


def incremental_learning(data, model):
    """Train LightGBM on a chunk, preserving the original workflow."""

    add_travel_vector_features(data)

    time_dummies = timeExpansion(data["pickup_datetime"])
    # Drop key safely – it may be absent when we skip loading it.
    X = pd.concat(
        [
            data.drop(
                ["key", "pickup_datetime", "fare_amount"], axis=1, errors="ignore"
            ).astype(np.float32),
            time_dummies,
        ],
        axis=1,
    )
    y = data["fare_amount"].astype(np.float32)

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

    params = {
        "boosting_type": "gbdt",
        "objective": "regression",
        "metric": "rmse",
        "num_leaves": 31,
        "learning_rate": 0.1,
        "feature_fraction": 0.9,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
        "verbose": -1,
        "n_jobs": -1,  # use all cores
    }

    dset = lgbm.Dataset(X_tr, label=y_tr, free_raw_data=True)
    model = lgbm.train(
        params,
        init_model=model,
        train_set=dset,
        keep_training_booster=True,
        num_boost_round=50,
    )
    mae = mean_absolute_error(y_te, model.predict(X_te))
    print("Mean Absolute Error (validation):", mae)
    return model




In [ ]:
def predict_and_submit(estimator, submit=True):
    test_df = pd.read_csv(
        "../input/test.csv",
        dtype={
            "key": "object",
            "pickup_longitude": np.float32,
            "pickup_latitude": np.float32,
            "dropoff_longitude": np.float32,
            "dropoff_latitude": np.float32,
            "passenger_count": np.int8,
        },
        parse_dates=["pickup_datetime"],
    )
    add_travel_vector_features(test_df)

    test_X = pd.concat(
        [
            test_df.drop(["key", "pickup_datetime"], axis=1).astype(np.float32),
            timeExpansion(test_df["pickup_datetime"]),
        ],
        axis=1,
    )

    print("Test Shape:", test_X.shape)

    test_y_predictions = estimator.predict(test_X)

    if submit:
        submission = pd.DataFrame(
            {"key": test_df.key, "fare_amount": test_y_predictions},
            columns=["key", "fare_amount"],
        )
        submission.to_csv("submission.csv", index=False)




In [ ]:
chunk_size = 20_000_000

origin_cols = [
    "fare_amount",
    "pickup_datetime",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "passenger_count",
]  # 'key' omitted – not needed for training

dtype_map = {
    "fare_amount": np.float32,
    "pickup_longitude": np.float32,
    "pickup_latitude": np.float32,
    "dropoff_longitude": np.float32,
    "dropoff_latitude": np.float32,
    "passenger_count": np.int8,
}

estimator = None
for i, chunk in enumerate(
    pd.read_csv(
        "../input/train.csv",
        chunksize=chunk_size,
        header=0,
        usecols=origin_cols,  # load only required columns
        dtype=dtype_map,
        parse_dates=["pickup_datetime"],
    )
):
    start = time.time()
    print(f"Chunk {i+1}: rows {i*chunk_size} to {(i+1)*chunk_size}")
    estimator = incremental_learning(chunk, estimator)
    print("Chunk", i + 1, "took", time.time() - start, "seconds\n")

predict_and_submit(estimator)